# Combinação do KNN K=11 com a árvore de quatro atributos

Combinamos os dois modelos que já enviamos ao Kaggle. A árvore teve pontuação pública de 0,83018 e o primeiro KNN K=11 teve 0,82897. Esses resultados ficam como registro das submissões individuais.

Os modelos mantêm suas próprias regras de preparação. O KNN usa os nove atributos numéricos e tabagismo, com mediana e quartis. A árvore usa pressão sistólica, glicose, hemoglobina e triglicerídeos, com preenchimento pela média e cortes por ganho de informação.

A combinação é uma proposta nossa. Os materiais ensinam KNN e árvore separadamente, e a regra de juntar as duas respostas é uma decisão deste experimento. Usamos NumPy para as contas da árvore, implementada do zero como no notebook anterior. As saídas estão salvas para leitura.

### Resultado desta comparação

A melhor das três regras testadas deu 50% de peso ao apoio do KNN e 50% ao apoio da árvore. A regra usa a média das duas frações para decidir a classe, com empate resolvido pelo KNN.

| Avaliação | KNN K=11 | Árvore | Mistura 50% / 50% |
|---|---:|---:|---:|
| Acurácia média nas dez rodadas usadas na escolha | 86,66% | 90,39% | 91,11% |
| F1 macro médio nas mesmas rodadas | 86,63% | 90,37% | 91,09% |
| Acurácia na conferência com semente 101 | 86,07% | 88,24% | 89,21% |
| Acurácia nas 104 amostras reservadas | 86,54% | 84,62% | 87,50% |
| Acertos nas 104 amostras reservadas | 90 | 88 | 91 |

A mistura corrigiu sete erros do KNN e perdeu seis acertos, com saldo de um acerto a mais. O ganho na reserva foi pequeno, e os resultados das avaliações devem ser apresentados separadamente.

O CSV da mistura foi gerado com os dois modelos ajustados nas 520 amostras rotuladas. Antes de combiná-los, conferimos que suas previsões individuais eram idênticas às duas submissões originais.

#### Resultado no Kaggle

O arquivo `submissao_knn11_arvore.csv` recebeu **0,84382** no placar público, conforme a captura enviada pelo aluno. A pontuação superou o primeiro KNN K=11 (0,82897) e a árvore separada (0,83018), mas ficou abaixo da meta de 0,87. Corresponde à mistura original, com metade do peso para cada modelo e limite de 50%. Não é o resultado da rede neural.


### Como juntar as respostas

O KNN fornece a fração dos 11 vizinhos que votaram em diabetes. Se sete vizinhos votam em diabetes, seu apoio à classe 1 é 7/11.

A árvore fornece a fração de exemplos de diabetes na folha alcançada. Se a folha tem 20 exemplos e 15 são de diabetes, o apoio é 15/20. Usamos a árvore compacta com as mesmas previsões de classe da versão enviada.

Comparamos três misturas: metade para cada modelo, 25% para a árvore e 75% para o KNN, ou 75% para a árvore e 25% para o KNN. A conta é uma média ponderada dos dois apoios. Esses números são frações observadas no treinamento, não probabilidades clínicas calibradas.

Quando o apoio combinado passa de 0,5, prevemos diabetes. Abaixo de 0,5, prevemos controle. Em empate exato, usamos a classe do KNN. Os modelos individuais também são avaliados para verificar se a mistura ajuda.

### Base e divisões

Repetimos a divisão com semente 42, com 416 amostras para escolher a mistura e 104 reservadas para comparação local. Escolhemos o peso pelo F1 macro médio em cinco partes com semente 42 e cinco com semente 73. Depois conferimos a mistura escolhida com semente 101.

Os modelos e as divisões já participaram de experiências anteriores. Registramos a reserva de 104 como referência local já consultada.

In [1]:
from pathlib import Path
from statistics import median, quantiles
from time import perf_counter
import csv
import hashlib
import json
import math
import random

import numpy as np

raiz = Path.cwd()
if not (raiz / "dados" / "treinamento_padronizado.csv").is_file():
    raiz = raiz.parent
caminho_base = raiz / "dados" / "treinamento_padronizado.csv"
with caminho_base.open(encoding="utf-8-sig", newline="") as arquivo:
    amostras = list(csv.DictReader(arquivo))

numericas = (
    "AGE", "HEIGHT", "WEIGHT", "SYSTOLIC", "DIASTOLIC", "GLUCOSE",
    "HEMOGLOBIN", "TRIGLYCERIDES", "SODIUM",
)
indices_arvore = tuple(numericas.index(c) for c in ("SYSTOLIC", "GLUCOSE", "HEMOGLOBIN", "TRIGLYCERIDES"))
classes = np.array([int(a["class"]) for a in amostras], dtype=np.int8)
pacientes = [a["PATIENT_CODE"].strip() for a in amostras]
if any(not p for p in pacientes) or len(set(pacientes)) != len(pacientes):
    raise ValueError("Conferir os registros de paciente.")
if not np.all(np.isin(classes, [0, 1])):
    raise ValueError("Esperávamos classes 0 e 1.")

controle = np.flatnonzero(classes == 0).tolist()
diabetes = np.flatnonzero(classes == 1).tolist()
gerador = random.Random(42)
gerador.shuffle(controle)
gerador.shuffle(diabetes)
n0, n1 = round(len(controle) * 0.20), round(len(diabetes) * 0.20)
reservadas = np.array(controle[:n0] + diabetes[:n1])
treinamento = np.array(controle[n0:] + diabetes[n1:])


def dividir(indices, semente):
    partes = [[] for _ in range(5)]
    gerador = random.Random(semente)
    for classe in (0, 1):
        grupo = [int(i) for i in indices if classes[i] == classe]
        gerador.shuffle(grupo)
        for posicao, i in enumerate(grupo):
            partes[posicao % 5].append(i)
    for posicao, parte in enumerate(partes):
        referencia = [i for j, outra in enumerate(partes) if j != posicao for i in outra]
        yield np.array(referencia), np.array(parte)


def ler_numero(amostra, coluna):
    try:
        valor = float(str(amostra[coluna]).strip().replace(",", "."))
    except ValueError:
        return None
    if not math.isfinite(valor) or valor <= 0:
        return None
    if coluna == "HEMOGLOBIN" and valor > 100:
        return None
    return valor


print(f"Treinamento: {len(treinamento)} | Reserva: {len(reservadas)}")
print("SHA-256 da base:", hashlib.sha256(caminho_base.read_bytes()).hexdigest())

Treinamento: 416 | Reserva: 104
SHA-256 da base: bc1a686d83c06e2d7667a6089e82b54b4bff2ca6467b81155f1246ac4719a032


### Resposta do KNN original

Reproduzimos o primeiro KNN K=11. Em cada atributo numérico, preenchemos os valores ausentes pela mediana e dividimos a diferença pela distância entre o primeiro e o terceiro quartil do treinamento.

A distância é Manhattan ponderada, com peso 2 para glicose e hemoglobina e peso 1 para as demais medidas. Tiramos a média ponderada e aplicamos D / (1 + D). Quando o tabagismo pode ser comparado, ele recebe 10% da distância final. Para um único atributo categórico, a distância de Jaccard é 0 para categorias iguais e 1 para categorias diferentes.

Tabagismo ausente ou inédito fica fora da comparação naquela dupla. Ordenamos as distâncias e contamos quantos dos 11 primeiros vizinhos são da classe 1. As distâncias iguais preservam a ordem embaralhada com semente 42.

In [2]:
pesos_knn = [2 if c in {"GLUCOSE", "HEMOGLOBIN"} else 1 for c in numericas]


def fumante(amostra):
    valor = amostra["SMOKER"].strip()
    return "" if valor in {"", "desconhecido"} else valor


def preparar_knn(referencia):
    parametros = {}
    for coluna in numericas:
        valores = [ler_numero(a, coluna) for a in referencia]
        disponiveis = [v for v in valores if v is not None]
        if len(disponiveis) < 2:
            raise ValueError(f"Valores insuficientes em {coluna}.")
        q1, _, q3 = quantiles(disponiveis, n=4, method="inclusive")
        if q3 == q1:
            raise ValueError(f"Intervalo entre quartis igual a zero em {coluna}.")
        parametros[coluna] = (median(disponiveis), q3 - q1)
    return parametros


def normalizar_knn(amostra, parametros):
    resultado = []
    for coluna in numericas:
        valor = ler_numero(amostra, coluna)
        mediana, intervalo = parametros[coluna]
        if valor is None:
            valor = mediana
        resultado.append((valor - mediana) / intervalo)
    return resultado


def apoiar_knn(referencia, consulta):
    parametros = preparar_knn(referencia)
    categorias_conhecidas = {fumante(a) for a in referencia if fumante(a)}
    vizinhos_referencia = [
        (normalizar_knn(a, parametros), fumante(a), int(a["class"]))
        for a in referencia
    ]
    random.Random(42).shuffle(vizinhos_referencia)
    apoios = []

    for amostra in consulta:
        numeros = normalizar_knn(amostra, parametros)
        categoria = fumante(amostra)
        vizinhos = []

        for numeros_ref, categoria_ref, classe in vizinhos_referencia:
            soma = 0.0
            for j, peso in enumerate(pesos_knn):
                soma += peso * abs(numeros[j] - numeros_ref[j])
            media = soma / sum(pesos_knn)
            distancia = media / (1 + media)
            if categoria and categoria_ref and categoria in categorias_conhecidas:
                jaccard = 0.0 if categoria == categoria_ref else 1.0
                distancia = 0.90 * distancia + 0.10 * jaccard
            vizinhos.append((distancia, classe))

        vizinhos.sort(key=lambda vizinho: vizinho[0])
        votos_diabetes = sum(classe for distancia, classe in vizinhos[:11])
        apoios.append(votos_diabetes / 11)

    apoios = np.array(apoios)
    return apoios, (apoios > 0.5).astype(np.int8)

### Resposta da árvore enviada

Reutilizamos a construção por ganho de informação da árvore de quatro atributos. Cada referência usa suas próprias médias para preencher ausências, mínimo de 15 exemplos por folha e profundidade máxima 6.

Juntamos caminhos que dão a mesma classe para obter a árvore compacta. Além da classe, cada folha guarda a quantidade de exemplos que chegaram até ela e quantos são de diabetes. A fração positiva dessa folha entra na mistura.

In [3]:
def entropia(positivos, total):
    proporcao = np.asarray(positivos, dtype=float) / total
    p0 = 1 - proporcao
    parte1 = proporcao * np.log2(np.maximum(proporcao, 1e-15))
    parte0 = p0 * np.log2(np.maximum(p0, 1e-15))
    return -(parte1 + parte0)


def melhor_teste(x, y, atributos, minimo_folha):
    total = len(y)
    positivos = int(y.sum())
    antes = float(entropia(positivos, total))
    melhor = None
    melhor_ganho = 1e-12

    for j in atributos:
        valores = x[:, j]
        if j < len(numericas):
            ordem = np.argsort(valores, kind="stable")
            ordenados = valores[ordem]
            quantidades = np.arange(1, total)
            positivos_esquerda = np.cumsum(y[ordem])[:-1]
            candidatos = (
                (ordenados[:-1] < ordenados[1:])
                & (quantidades >= minimo_folha)
                & (total - quantidades >= minimo_folha)
            )
            if not candidatos.any():
                continue
            posicoes = np.flatnonzero(candidatos)
            n_esq = quantidades[posicoes]
            p_esq = positivos_esquerda[posicoes]
            depois = (
                n_esq * entropia(p_esq, n_esq)
                + (total - n_esq) * entropia(positivos - p_esq, total - n_esq)
            ) / total
            ganhos = antes - depois
            k = int(np.argmax(ganhos))
            ganho = float(ganhos[k])
            posicao = int(posicoes[k])
            limite = float((ordenados[posicao] + ordenados[posicao + 1]) / 2)
            tipo = "numero"
        else:
            unicos = np.unique(valores)
            ganhos = []
            limites = []
            for valor in unicos:
                esquerda = valores == valor
                n_esq = int(esquerda.sum())
                if n_esq < minimo_folha or total - n_esq < minimo_folha:
                    continue
                p_esq = int(y[esquerda].sum())
                depois = (
                    n_esq * entropia(p_esq, n_esq)
                    + (total - n_esq) * entropia(positivos - p_esq, total - n_esq)
                ) / total
                ganhos.append(float(antes - depois))
                limites.append(float(valor))
            if not ganhos:
                continue
            k = int(np.argmax(ganhos))
            ganho, limite = ganhos[k], limites[k]
            tipo = "categoria"

        if ganho > melhor_ganho:
            melhor_ganho = ganho
            melhor = (j, tipo, limite, ganho)
    return melhor

In [4]:
def construir_arvore(x, y, atributos, minimo_folha, padrao=0):
    positivos = int(y.sum())
    if positivos * 2 == len(y):
        classe = padrao
    else:
        classe = int(positivos * 2 > len(y))
    no = {"classe": classe, "amostras": len(y), "positivos": positivos}
    if positivos == 0 or positivos == len(y) or len(y) < 2 * minimo_folha:
        return no

    teste = melhor_teste(x, y, atributos, minimo_folha)
    if teste is None:
        return no
    j, tipo, limite, ganho = teste
    esquerda = x[:, j] <= limite if tipo == "numero" else x[:, j] == limite
    no.update(atributo=j, tipo=tipo, limite=limite, ganho=ganho)
    if tipo == "categoria":
        no["conhecidas"] = np.unique(x[:, j]).tolist()
    no["sim"] = construir_arvore(x[esquerda], y[esquerda], atributos, minimo_folha, classe)
    no["nao"] = construir_arvore(x[~esquerda], y[~esquerda], atributos, minimo_folha, classe)
    return no


def simplificar_arvore(no, profundidade, nivel=0):
    folha = {
        "classe": no["classe"],
        "amostras": no["amostras"],
        "positivos": no["positivos"],
    }
    if "atributo" not in no or (profundidade is not None and nivel >= profundidade):
        return folha

    copia = no.copy()
    copia["sim"] = simplificar_arvore(no["sim"], profundidade, nivel + 1)
    copia["nao"] = simplificar_arvore(no["nao"], profundidade, nivel + 1)
    sim, nao = copia["sim"], copia["nao"]

    if "atributo" not in sim and "atributo" not in nao and sim["classe"] == nao["classe"]:
        if no["tipo"] == "numero" or sim["classe"] == no["classe"]:
            folha["classe"] = sim["classe"]
            return folha
    return copia

In [5]:
def matriz_arvore(amostras_entrada, medias=None):
    numeros = np.array([
        [ler_numero(a, c) for c in numericas] for a in amostras_entrada
    ], dtype=float)
    if medias is None:
        medias = []
        for j, coluna in enumerate(numericas):
            disponiveis = numeros[np.isfinite(numeros[:, j]), j]
            if not len(disponiveis):
                raise ValueError(f"Nenhuma medida disponível em {coluna}.")
            medias.append(float(np.mean(disponiveis)))
    for j, valor in enumerate(medias):
        numeros[:, j] = np.where(np.isfinite(numeros[:, j]), numeros[:, j], valor)
    return numeros, medias


def folha_alcancada(entrada, arvore):
    no = arvore
    while "atributo" in no:
        caminho = "sim" if entrada[no["atributo"]] <= no["limite"] else "nao"
        no = no[caminho]
    return no


def apoiar_arvore(referencia, consulta):
    x, medias = matriz_arvore(referencia)
    aval, _ = matriz_arvore(consulta, medias)
    y = np.array([int(a["class"]) for a in referencia], dtype=np.int8)
    arvore = construir_arvore(x, y, indices_arvore, 15)
    arvore = simplificar_arvore(arvore, 6)
    folhas = [folha_alcancada(a, arvore) for a in aval]
    apoios = np.array([no["positivos"] / no["amostras"] for no in folhas])
    previstas = np.array([no["classe"] for no in folhas], dtype=np.int8)
    return apoios, previstas


def combinar(apoio_knn, apoio_arvore, peso_arvore):
    apoio = (1 - peso_arvore) * apoio_knn + peso_arvore * apoio_arvore
    previstas = (apoio > 0.5).astype(np.int8)
    empate = np.isclose(apoio, 0.5, rtol=0, atol=1e-12)
    previstas[empate] = (apoio_knn[empate] > 0.5).astype(np.int8)
    return previstas


def contar(reais, previstas):
    return np.array([
        np.sum((reais == 0) & (previstas == 0)),
        np.sum((reais == 0) & (previstas == 1)),
        np.sum((reais == 1) & (previstas == 0)),
        np.sum((reais == 1) & (previstas == 1)),
    ])


def razao(a, b):
    return np.divide(a, b, out=np.zeros_like(a, dtype=float), where=b != 0)


def metricas(contagens):
    tn, fp, fn, tp = np.moveaxis(np.asarray(contagens), -1, 0)
    return {
        "acuracia": razao(tn + tp, tn + fp + fn + tp),
        "f1": (razao(2 * tn, 2 * tn + fp + fn) + razao(2 * tp, 2 * tp + fp + fn)) / 2,
        "precisao": (razao(tn, tn + fn) + razao(tp, tp + fp)) / 2,
        "recall0": razao(tn, tn + fp),
        "recall1": razao(tp, tp + fn),
    }


def mostrar(nome, contagens):
    m = metricas(contagens)
    print(f"{nome:24} | " + " | ".join(
        f"{np.mean(m[c]):>8.2%}" for c in ("acuracia", "f1", "precisao", "recall0", "recall1")
    ))

### Escolha do peso da mistura

Testamos peso da árvore igual a 50%, 25% ou 75%. O KNN recebe o restante. Escolhemos pelo maior F1 macro médio nas dez rodadas das sementes 42 e 73. Em empate exato, preferimos 50% por simplicidade, seguindo depois a ordem declarada de 25% e 75%.

Recalculamos cada modelo com a parte de referência da rodada. Assim, as medianas do KNN, as médias da árvore e as duas decisões são produzidas usando apenas aquela referência.

In [6]:
pesos_arvore = (0.50, 0.25, 0.75)
nomes = ["KNN K=11", "Árvore"] + [f"Mistura árvore {peso:.0%}" for peso in pesos_arvore]
rodadas = []
inicio = perf_counter()

for semente in (42, 73):
    for numero, (indices_ref, indices_aval) in enumerate(dividir(treinamento, semente), start=1):
        referencia = [amostras[i] for i in indices_ref]
        consulta = [amostras[i] for i in indices_aval]
        apoio_knn, classe_knn = apoiar_knn(referencia, consulta)
        apoio_arvore, classe_arvore = apoiar_arvore(referencia, consulta)
        contagens = [
            contar(classes[indices_aval], classe_knn),
            contar(classes[indices_aval], classe_arvore),
        ]
        for peso in pesos_arvore:
            contagens.append(contar(
                classes[indices_aval], combinar(apoio_knn, apoio_arvore, peso)
            ))
        rodadas.append(contagens)

rodadas = np.array(rodadas)
print("Versão                   | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1")
for j, nome in enumerate(nomes):
    mostrar(nome, rodadas[:, j])

f1_misturas = metricas(rodadas[:, 2:])["f1"].mean(axis=0)
melhor_peso = pesos_arvore[int(np.argmax(f1_misturas))]
print(f"Peso escolhido para a árvore: {melhor_peso:.0%}")
print(f"Tempo das dez rodadas: {perf_counter() - inicio:.1f} s")

Versão                   | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1
KNN K=11                 |   86.66% |   86.63% |   87.28% |   92.75% |   81.00%
Árvore                   |   90.39% |   90.37% |   90.51% |   91.50% |   89.36%
Mistura árvore 50%       |   91.11% |   91.09% |   91.32% |   93.25% |   89.12%
Mistura árvore 25%       |   90.51% |   90.50% |   90.85% |   94.75% |   86.56%
Mistura árvore 75%       |   90.63% |   90.61% |   90.76% |   91.75% |   89.59%
Peso escolhido para a árvore: 50%
Tempo das dez rodadas: 0.7 s


### Conferência com semente 101 e nas 104 amostras

Fixamos o peso escolhido e repetimos a avaliação nas cinco partes da semente 101. Depois usamos as 416 amostras como referência para os dois modelos e prevemos as 104 reservadas.

Também contamos os erros corrigidos e os acertos perdidos em relação ao KNN original. Essas contagens mostram o efeito da regra de combinação sobre a mesma lista de amostras.

In [7]:
conferencia = []
for indices_ref, indices_aval in dividir(treinamento, 101):
    referencia = [amostras[i] for i in indices_ref]
    consulta = [amostras[i] for i in indices_aval]
    apoio_knn, classe_knn = apoiar_knn(referencia, consulta)
    apoio_arvore, classe_arvore = apoiar_arvore(referencia, consulta)
    classe_mistura = combinar(apoio_knn, apoio_arvore, melhor_peso)
    conferencia.append([
        contar(classes[indices_aval], classe_knn),
        contar(classes[indices_aval], classe_arvore),
        contar(classes[indices_aval], classe_mistura),
    ])
conferencia = np.array(conferencia)

print("Conferência, semente 101")
print("Versão                   | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1")
for j, nome in enumerate(("KNN K=11", "Árvore", "Mistura escolhida")):
    mostrar(nome, conferencia[:, j])

referencia = [amostras[i] for i in treinamento]
consulta = [amostras[i] for i in reservadas]
apoio_knn_reserva, classe_knn_reserva = apoiar_knn(referencia, consulta)
apoio_arvore_reserva, classe_arvore_reserva = apoiar_arvore(referencia, consulta)
classe_mistura_reserva = combinar(apoio_knn_reserva, apoio_arvore_reserva, melhor_peso)
reais_reserva = classes[reservadas]
reserva = np.array([
    contar(reais_reserva, classe_knn_reserva),
    contar(reais_reserva, classe_arvore_reserva),
    contar(reais_reserva, classe_mistura_reserva),
])

print()
print("Nas 104 amostras reservadas")
print("Versão                   | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1")
for j, nome in enumerate(("KNN K=11", "Árvore", "Mistura escolhida")):
    mostrar(nome, reserva[j])
    print("Matriz:", reserva[j].reshape(2, 2).tolist())

corrigidos = int(np.sum((classe_knn_reserva != reais_reserva) & (classe_mistura_reserva == reais_reserva)))
perdidos = int(np.sum((classe_knn_reserva == reais_reserva) & (classe_mistura_reserva != reais_reserva)))
print("Erros do KNN corrigidos pela mistura:", corrigidos)
print("Acertos do KNN perdidos pela mistura:", perdidos)

Conferência, semente 101
Versão                   | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1
KNN K=11                 |   86.07% |   86.02% |   86.86% |   91.50% |   81.04%
Árvore                   |   88.24% |   88.23% |   88.36% |   90.00% |   86.64%
Mistura escolhida        |   89.21% |   89.19% |   89.51% |   92.50% |   86.19%

Nas 104 amostras reservadas
Versão                   | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1
KNN K=11                 |   86.54% |   86.53% |   86.54% |   88.00% |   85.19%
Matriz: [[44, 6], [8, 46]]
Árvore                   |   84.62% |   84.61% |   84.62% |   86.00% |   83.33%
Matriz: [[43, 7], [9, 45]]
Mistura escolhida        |   87.50% |   87.49% |   87.48% |   88.00% |   87.04%
Matriz: [[44, 6], [7, 47]]
Erros do KNN corrigidos pela mistura: 7
Acertos do KNN perdidos pela mistura: 6


### Geração e conferência do CSV

Antes de exportar a mistura, refazemos os dois modelos com as 520 amostras rotuladas. Aplicamos as mesmas regras de padronização numérica ao teste e traduzimos tabagismo como na exploração.

Conferimos se cada modelo sozinho reproduz exatamente suas 312 previsões já enviadas. Depois geramos a mistura, mantendo a ordem dos IDs. Essa conferência garante que estamos combinando as duas versões solicitadas.

In [8]:
def padronizar_teste(amostra):
    entrada = amostra.copy()
    if entrada["AGE"].strip() in {"0", "-1", "999"}:
        entrada["AGE"] = ""
    altura = ler_numero(entrada, "HEIGHT")
    if altura is not None and altura < 3:
        entrada["HEIGHT"] = str(altura * 100)
    pressao = entrada["BLOOD_PRESSURE"].strip().replace("x", "/").replace(" ", "")
    if pressao in {"", "?"}:
        entrada["SYSTOLIC"], entrada["DIASTOLIC"] = "", ""
    else:
        sistolica, diastolica = pressao.split("/")
        entrada["SYSTOLIC"] = str(int(sistolica))
        entrada["DIASTOLIC"] = str(int(diastolica))
    mapa = {
        "never": "nunca_fumou", "former": "ex_fumante", "ex-smoker": "ex_fumante",
        "current": "fumante_atual", "yes": "fumante_atual", "": "",
    }
    valor = entrada["SMOKER"].strip().lower()
    entrada["SMOKER"] = mapa.get(valor, valor)
    return entrada


with (raiz / "dados" / "kaggle-test" / "test.csv").open(
    encoding="utf-8-sig", newline=""
) as arquivo:
    teste_original = list(csv.DictReader(arquivo))
ids_teste = [a["id"] for a in teste_original]
if not ids_teste or len(set(ids_teste)) != len(ids_teste) or any(not i.strip() for i in ids_teste):
    raise ValueError("O teste precisa ter IDs únicos e preenchidos.")
teste = [padronizar_teste(a) for a in teste_original]

apoio_knn_teste, classe_knn_teste = apoiar_knn(amostras, teste)
apoio_arvore_teste, classe_arvore_teste = apoiar_arvore(amostras, teste)


def conferir_submissao(caminho, previstas):
    with caminho.open(encoding="utf-8-sig", newline="") as arquivo:
        leitor = csv.DictReader(arquivo)
        if leitor.fieldnames != ["id", "class"]:
            raise ValueError(f"Cabeçalho inesperado em {caminho.name}.")
        linhas = list(leitor)
    if [a["id"] for a in linhas] != ids_teste:
        raise ValueError(f"IDs diferentes em {caminho.name}.")
    if [int(a["class"]) for a in linhas] != previstas.tolist():
        raise ValueError(f"As previsões de {caminho.name} não foram reproduzidas.")


conferir_submissao(raiz / "resultados" / "submissao_knn_k11.csv", classe_knn_teste)
conferir_submissao(raiz / "resultados" / "submissao_arvore_4_atributos.csv", classe_arvore_teste)
print("KNN e árvore reproduzem as submissões individuais, linha por linha.")
print("Previsões por modelo:", len(ids_teste))

KNN e árvore reproduzem as submissões individuais, linha por linha.
Previsões por modelo: 312


In [9]:
classe_mistura_teste = combinar(apoio_knn_teste, apoio_arvore_teste, melhor_peso)
caminho_envio = raiz / "resultados" / "submissao_knn11_arvore.csv"
with caminho_envio.open("w", encoding="utf-8", newline="") as arquivo:
    escritor = csv.writer(arquivo)
    escritor.writerow(["id", "class"])
    escritor.writerows(zip(ids_teste, classe_mistura_teste.tolist(), strict=True))
conferir_submissao(caminho_envio, classe_mistura_teste)

caminho_registro = raiz / "resultados" / "knn11_arvore.npz"
np.savez_compressed(
    caminho_registro, rodadas=rodadas, conferencia=conferencia, reserva=reserva,
    apoio_knn_teste=apoio_knn_teste, apoio_arvore_teste=apoio_arvore_teste,
    peso_arvore=melhor_peso,
    catalogo=json.dumps({
        "nomes": nomes, "sementes_escolha": [42, 73], "semente_conferencia": 101,
        "pesos_arvore": pesos_arvore, "peso_escolhido": melhor_peso,
        "hash_base": hashlib.sha256(caminho_base.read_bytes()).hexdigest(),
        "knn_publico": 0.82897, "arvore_publico": 0.83018,
    }),
)
print("CSV salvo:", caminho_envio.relative_to(raiz))
print("Linhas:", len(classe_mistura_teste))
print("Previsões diferentes do KNN:", int(np.sum(classe_mistura_teste != classe_knn_teste)))
print("Previsões diferentes da árvore:", int(np.sum(classe_mistura_teste != classe_arvore_teste)))
print("SHA-256 do CSV:", hashlib.sha256(caminho_envio.read_bytes()).hexdigest())

CSV salvo: resultados\submissao_knn11_arvore.csv
Linhas: 312
Previsões diferentes do KNN: 37
Previsões diferentes da árvore: 10
SHA-256 do CSV: f3ac1444e3cbb1d159d1798cd7f914068b205e77aa711d19b8bc1d2ffd57a6b6


### Ajustes na regra de combinação

Mantemos o primeiro KNN K=11 e a árvore de quatro atributos exatamente como antes. Procuramos uma regra de combinação que melhore o equilíbrio entre as duas classes.

Comparamos pesos intermediários da média e limites de 45%, 50% e 55% para prever diabetes. Também testamos regras de preferência. Em uma delas, usamos o KNN se a classe mais votada tiver pelo menos oito, nove ou dez dos 11 vizinhos. Nos outros casos, usamos a árvore. Em outra, usamos a árvore se a folha tiver pelo menos 65%, 75% ou 85% dos exemplos de uma mesma classe. Nos outros casos, usamos o KNN.

Finalmente, testamos usar a mistura de metade para cada quando os dois exames estão disponíveis, ou quando há pelo menos um disponível, recorrendo ao KNN nos demais casos.

São propostas deste experimento. As frações de apoio são contagens dos modelos, não probabilidades clínicas calibradas. Os limites e as condições são definidos antes de comparar as regras.

In [10]:
regras_ajuste = [
    {"nome": "Média 50/50, limite 50%", "tipo": "media", "peso": 0.50, "limite": 0.50}
]
for peso in (0.25, 0.375, 0.50, 0.625, 0.75):
    for limite in (0.45, 0.50, 0.55):
        regra = {
            "nome": f"Árvore {peso:.1%}, limite {limite:.0%}",
            "tipo": "media", "peso": peso, "limite": limite,
        }
        if (peso, limite) != (0.50, 0.50):
            regras_ajuste.append(regra)
for peso in (0.10, 0.90):
    regras_ajuste.append({
        "nome": f"Árvore {peso:.0%}, limite 50%",
        "tipo": "media", "peso": peso, "limite": 0.50,
    })
for quantidade in (8, 9, 10):
    regras_ajuste.append({
        "nome": f"KNN com pelo menos {quantidade} votos",
        "tipo": "knn_forte", "votos": quantidade,
    })
for apoio in (0.65, 0.75, 0.85):
    regras_ajuste.append({
        "nome": f"Árvore com pelo menos {apoio:.0%}",
        "tipo": "arvore_forte", "apoio": apoio,
    })
for observados in (2, 1):
    regras_ajuste.append({
        "nome": f"Mistura com pelo menos {observados} exame(s)",
        "tipo": "exames", "observados": observados,
    })


def aplicar_regra(regra, apoio_knn, classe_knn, apoio_arvore, classe_arvore, consulta):
    if regra["tipo"] == "media":
        apoio = (1 - regra["peso"]) * apoio_knn + regra["peso"] * apoio_arvore
        previstas = (apoio > regra["limite"]).astype(np.int8)
        empate = np.isclose(apoio, regra["limite"], rtol=0, atol=1e-12)
        previstas[empate] = classe_knn[empate]
        return previstas
    if regra["tipo"] == "knn_forte":
        votos = np.rint(11 * apoio_knn).astype(int)
        maioria = np.maximum(votos, 11 - votos)
        return np.where(maioria >= regra["votos"], classe_knn, classe_arvore)
    if regra["tipo"] == "arvore_forte":
        apoio_maioria = np.maximum(apoio_arvore, 1 - apoio_arvore)
        return np.where(apoio_maioria >= regra["apoio"], classe_arvore, classe_knn)

    disponiveis = np.array([
        sum(ler_numero(a, coluna) is not None for coluna in ("GLUCOSE", "HEMOGLOBIN"))
        for a in consulta
    ])
    mistura = combinar(apoio_knn, apoio_arvore, 0.50)
    return np.where(disponiveis >= regra["observados"], mistura, classe_knn)


def prever_modelos(indices_ref, indices_aval):
    referencia = [amostras[i] for i in indices_ref]
    consulta = [amostras[i] for i in indices_aval]
    apoio_knn, classe_knn = apoiar_knn(referencia, consulta)
    apoio_arvore, classe_arvore = apoiar_arvore(referencia, consulta)
    return apoio_knn, classe_knn, apoio_arvore, classe_arvore, consulta


resultados_ajuste = []
for semente in (42, 73):
    for indices_ref, indices_aval in dividir(treinamento, semente):
        informacoes = prever_modelos(indices_ref, indices_aval)
        resultados_ajuste.append([
            contar(classes[indices_aval], aplicar_regra(regra, *informacoes))
            for regra in regras_ajuste
        ])
resultados_ajuste = np.array(resultados_ajuste)

f1_ajuste = metricas(resultados_ajuste)["f1"].mean(axis=0)
ordem_ajuste = sorted(range(len(regras_ajuste)), key=lambda i: (-f1_ajuste[i], i))
indice_ajuste = ordem_ajuste[0]
regra_ajustada = regras_ajuste[indice_ajuste]

print("Regras comparadas:", len(regras_ajuste))
print("Dez melhores regras nas divisões de escolha")
for i in ordem_ajuste[:10]:
    m = metricas(resultados_ajuste[:, i])
    print(
        f"Acc {np.mean(m['acuracia']):.2%} | F1 {np.mean(m['f1']):.2%} | "
        f"Recall 0 {np.mean(m['recall0']):.2%} | Recall 1 {np.mean(m['recall1']):.2%} | "
        f"{regras_ajuste[i]['nome']}"
    )
print("Regra escolhida:", regra_ajustada["nome"])

Regras comparadas: 25
Dez melhores regras nas divisões de escolha
Acc 91.95% | F1 91.93% | Recall 0 92.25% | Recall 1 91.67% | Árvore 50.0%, limite 45%
Acc 91.23% | F1 91.21% | Recall 0 91.75% | Recall 1 90.75% | Árvore 62.5%, limite 45%
Acc 91.11% | F1 91.10% | Recall 0 93.75% | Recall 1 88.66% | Árvore 37.5%, limite 50%
Acc 91.11% | F1 91.09% | Recall 0 93.25% | Recall 1 89.12% | Média 50/50, limite 50%
Acc 91.11% | F1 91.08% | Recall 0 91.25% | Recall 1 90.97% | Árvore com pelo menos 75%
Acc 90.99% | F1 90.97% | Recall 0 91.75% | Recall 1 90.28% | Árvore com pelo menos 65%
Acc 90.87% | F1 90.86% | Recall 0 92.75% | Recall 1 89.12% | KNN com pelo menos 10 votos
Acc 90.87% | F1 90.85% | Recall 0 92.00% | Recall 1 89.82% | Árvore 62.5%, limite 50%
Acc 90.87% | F1 90.85% | Recall 0 91.50% | Recall 1 90.29% | Árvore 75.0%, limite 45%
Acc 90.87% | F1 90.85% | Recall 0 93.00% | Recall 1 88.89% | Árvore com pelo menos 85%
Regra escolhida: Árvore 50.0%, limite 45%


#### Conferência da regra escolhida

Fixamos a regra pelo resultado das sementes 42 e 73. Depois comparamos com a mistura original nas sementes 101, 21 e 145. Todas usam as mesmas 416 amostras em divisões diferentes, por isso medem a sensibilidade ao embaralhamento.

Ao final, aplicamos a regra escolhida às 104 amostras reservadas e contamos quantos erros da mistura original ela corrige e quantos acertos ela perde.

In [11]:
conferencias_ajuste = {}
for semente in (101, 21, 145):
    rodadas_conferencia = []
    for indices_ref, indices_aval in dividir(treinamento, semente):
        informacoes = prever_modelos(indices_ref, indices_aval)
        rodadas_conferencia.append([
            contar(classes[indices_aval], aplicar_regra(regra, *informacoes))
            for regra in (regras_ajuste[0], regra_ajustada)
        ])
    conferencias_ajuste[semente] = np.array(rodadas_conferencia)

print("Avaliação                | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1")
for semente, rodadas_conferencia in conferencias_ajuste.items():
    mostrar(f"Original, s{semente}", rodadas_conferencia[:, 0])
    mostrar(f"Ajustada, s{semente}", rodadas_conferencia[:, 1])

info_reserva = prever_modelos(treinamento, reservadas)
previstas_original = aplicar_regra(regras_ajuste[0], *info_reserva)
previstas_ajustada = aplicar_regra(regra_ajustada, *info_reserva)
reserva_ajuste = np.array([
    contar(classes[reservadas], previstas_original),
    contar(classes[reservadas], previstas_ajustada),
])
print()
print("Nas 104 amostras reservadas")
mostrar("Mistura original", reserva_ajuste[0])
mostrar("Mistura ajustada", reserva_ajuste[1])
print("Matriz original:", reserva_ajuste[0].reshape(2, 2).tolist())
print("Matriz ajustada:", reserva_ajuste[1].reshape(2, 2).tolist())

reais = classes[reservadas]
corrigidos_ajuste = int(np.sum((previstas_original != reais) & (previstas_ajustada == reais)))
perdidos_ajuste = int(np.sum((previstas_original == reais) & (previstas_ajustada != reais)))
print("Erros da mistura original corrigidos:", corrigidos_ajuste)
print("Acertos da mistura original perdidos:", perdidos_ajuste)

Avaliação                | Acurácia | F1 macro | Precisão | Recall 0 | Recall 1
Original, s101           |   89.21% |   89.19% |   89.51% |   92.50% |   86.19%
Ajustada, s101           |   89.68% |   89.67% |   89.75% |   90.00% |   89.40%
Original, s21            |   90.88% |   90.85% |   91.02% |   91.50% |   90.29%
Ajustada, s21            |   91.12% |   91.08% |   91.27% |   90.50% |   91.68%
Original, s145           |   91.11% |   91.10% |   91.23% |   92.50% |   89.84%
Ajustada, s145           |   91.35% |   91.33% |   91.41% |   90.50% |   92.15%

Nas 104 amostras reservadas
Mistura original         |   87.50% |   87.49% |   87.48% |   88.00% |   87.04%
Mistura ajustada         |   84.62% |   84.56% |   84.67% |   82.00% |   87.04%
Matriz original: [[44, 6], [7, 47]]
Matriz ajustada: [[41, 9], [7, 47]]
Erros da mistura original corrigidos: 0
Acertos da mistura original perdidos: 3


In [12]:
caminho_registro_ajuste = raiz / "resultados" / "knn11_arvore_ajustes.npz"
np.savez_compressed(
    caminho_registro_ajuste,
    regras=json.dumps(regras_ajuste, ensure_ascii=False),
    busca=resultados_ajuste,
    escolhida=indice_ajuste,
    sementes_conferencia=np.array(list(conferencias_ajuste)),
    conferencias=np.array(list(conferencias_ajuste.values())),
    reserva=reserva_ajuste,
)
print("Registro dos ajustes:", caminho_registro_ajuste.relative_to(raiz))

Registro dos ajustes: resultados\knn11_arvore_ajustes.npz


#### Resultado dos ajustes

A melhor das 25 regras na seleção manteve a média de metade para cada modelo e reduziu o limite para prever diabetes de 50% para 45%.

A acurácia média nas divisões de escolha subiu de 91,11% para 91,95%. Nas conferências com sementes 101, 21 e 145, a acurácia também subiu. Entretanto, nas 104 amostras reservadas, a regra criou três novos erros de controle e não corrigiu erros de diabetes. A acurácia passou de 87,50% para 84,62%.

A regra com limite de 45% fica registrada como alternativa que favorece a identificação de diabetes nas divisões comparadas. A mistura original, com limite de 50%, continua sendo a referência local mais forte nas 104 amostras reservadas. Os CSVs anteriores preservam as regras com que foram gerados.


### Referências e versões combinadas

- [KNN original](../Mod1_KNN/02_Knn_numerico.ipynb), primeira submissão K=11, com 0,82897 no placar público.
- [Árvore de quatro atributos](../Mod2_AD/02_Arvore_busca_aulas.ipynb), com 0,83018 no placar público.
- [Aula 5](../docs/materiais_aula/aula5_MD.pdf), conceitos de KNN e árvore.
- [Aula 7](../docs/materiais_aula/aula7_MD.pdf), avaliação por divisões e métricas.

A regra de combinação por média ponderada das frações de apoio é uma proposta deste experimento. Mantemos no texto a distinção entre as duas implementações já submetidas e os algoritmos apresentados nas aulas.